Preparação do OpenDSS

In [2]:
import pandas as pd
import py_dss_interface
from py_dss_toolkit import dss_tools

dss_file = r"C:\Dados_teste\OpenDSS\GWO\Alim_Meia_Ponte_5_REDUZIDO\Master_PyDSS_Interface.dss"

dss = py_dss_interface.DSS()

dss_tools.update_dss(dss)

dss.text(f"compile [{dss_file}]")
dss.text("buscoords buscoords.csv")

def decide_element(bus: str, buses_df: pd.DataFrame, trafo_df: pd.DataFrame, lines_df: pd.DataFrame):
    bus = bus.split(".")[0]
    elements_query = buses_df.query(f"name == '{bus}'")["all_pde_active_bus"].iloc[0]

    distance = 1e6
    for element in elements_query:
        element_type, element_name = element.split(".")

        if element_type == "Line":
            segment_df = lines_df.query(f"name == '{element_name}'")

            element_bus = segment_df["bus_name"].iloc[0]
            distance_bus = buses_df.query(f"name == '{element_bus}'")["distance"].iloc[0]

            if "mt" in element_name:
                kv = 13.8

            elif "bt" in element_name:
                kv = 0.38

            else:
                continue

        elif element_type == "Transformer":
            element_bus = trafo_df.query(f"name == '{element_name}'")["bus_name"].iloc[0]
            distance_bus = buses_df.query(f"name == '{element_bus}'")["distance"].iloc[0]
            kv = trafo_df.query(f"name == '{element_name}'")["kv"].iloc[0]

        else:
            continue

        if distance_bus < distance:
            distance = distance_bus
            selected_element = element_name
            selected_element_type = element_type
            selected_kv = kv

    return selected_element_type, selected_element, selected_kv

def calc_rpf() -> float:
    soma = 0
    for name in dss.monitors.names:
        if "voltage" in name:
            continue

        if "battery" in name:
            continue

        dss.monitors.name = name
        monitor = dss.monitors.channel(1)

        soma += sum(x for x in monitor if x < 0)

    return soma

buses_df = dss_tools.model.buses_df
trafo_df = dss_tools.model.transformers_df
lines_df = dss_tools.model.lines_df

trafo_df["bus_name"] = trafo_df["bus"].str.split(".").str[0]
lines_df["bus_name"] = lines_df["bus1"].str.split(".").str[0]

buses_df = buses_df[["name", "distance", "all_pce_active_bus", "all_pde_active_bus"]]


In [22]:
import py_dss_interface
from py_dss_toolkit import dss_tools
from matplotlib import pyplot as plt

solution_buses = ['95217345002086b', '94336115002086b', '81489125002086', '82021615286802', '82025745286821']
#solution_buses = ['95217345002086b', '81489125002086', '82021615286802', '82025745286821']

dss_file = r"C:\Dados_teste\OpenDSS\GWO\Alim_Meia_Ponte_5_REDUZIDO\Master_PyDSS_Interface.dss"

dss = py_dss_interface.DSS()

dss_tools.update_dss(dss)

df_rpf = pd.DataFrame(columns=["Caso", "Fluxo Reverso Total (kWh)"])

voltage_dfs_base_case = {}

cases = ["Caso Base", "Caso Ótimo"]

for case in cases:

    dss.text(f"compile [{dss_file}]")

    dss.text("Redirect 'PV_System_120_MeiaPonte.dss'")
    #dss.text("Redirect 'PVSystem_120_Irrad.dss")
    dss.text("buscoords buscoords.csv")

    if case == "Caso Ótimo":

        for i, bus in enumerate(solution_buses):
            element_type, element_name, kv = decide_element(bus, buses_df, trafo_df, lines_df)
            pot = 280

            dss.text(f"New Storage.Battery_{i} phases=3 bus1={bus} kv={kv} kWrated={pot} kWhrated={8*pot} dispmode=follow %stored=50")
            
            dss.text(f"New StorageController.SC_{i} element={element_type}.{element_name} terminal=1 modedis=peakShave elementList=[Battery_{i}]\n"
            f"~ MonPhase=AVG kwtarget=5 modecharge=peakShaveLow kwtargetLow=0\n\n")

            dss.text(f"New Monitor.Battery_{i} element=Storage.Battery_{i} mode=7")

    dss.text(f"buscoords BusCoords.csv")

    dss.text("set mode=daily")
    dss.text("set stepsize=1h")
    dss.text("set number=24")

    dss.solution.solve()

    energymeter_df = dss_tools.results.energymeters

    if case == "Caso Base":
        for name in dss.monitors.names:
            if not "voltage" in name:
                continue

            voltage_dfs_base_case[name] = dss_tools.results.monitor(name)

    elem_1, elem_2, elem_3, elem_4, elem_5 = "trf_5286812a", "trf_5552819a", "trf_5286964a", "trf_5286821a", "trf_5286934a"

    if case == "Caso Ótimo":
        df_optimal_feeder = dss_tools.results.monitor("feeder")
        df_optimal_trafo_1 = dss_tools.results.monitor(elem_1)
        df_optimal_trafo_2 = dss_tools.results.monitor(elem_2)
        df_optimal_trafo_3 = dss_tools.results.monitor(elem_3)
        df_optimal_trafo_4 = dss_tools.results.monitor(elem_4)
        df_optimal_trafo_5 = dss_tools.results.monitor(elem_5)

    elif case == "Caso Base":
        df_base_feeder = dss_tools.results.monitor("feeder")
        df_base_trafo_1 = dss_tools.results.monitor(elem_1)
        df_base_trafo_2 = dss_tools.results.monitor(elem_2)
        df_base_trafo_3  = dss_tools.results.monitor(elem_3)
        df_base_trafo_4  = dss_tools.results.monitor(elem_4)
        df_base_trafo_5  = dss_tools.results.monitor(elem_5)

    print(f"{case}")
    display(energymeter_df[["kwh", "kvarh", "zone kwh", "zone losses kwh"]])

    rpf = calc_rpf()
    df2 = pd.DataFrame({"Caso": [case], "Fluxo Reverso Total (kWh)": [rpf]})

    df_rpf = pd.concat([df_rpf, df2], ignore_index=True)

#Fluxo reverso total
display(df_rpf)



Caso Base


,kwh,kvarh,zone kwh,zone losses kwh
0,49444.317416,65469.605055,120501.062097,5882.456263


Caso Ótimo


,kwh,kvarh,zone kwh,zone losses kwh
0,48330.814782,64691.205533,120571.610121,5528.233247


,Caso,Fluxo Reverso Total (kWh)
0,Caso Base,-23302.206599
1,Caso Ótimo,-21253.003351


Elementos nas barras problemáticas

In [44]:
buses_df = dss_tools.model.buses_df
solution_buses_fixed = []

for bus in solution_buses:
    solution_buses_fixed.append(bus.split(".")[0])

display(buses_df.query(f"name in {solution_buses_fixed}"))

,name,nodes,num_nodes,kv_base,distance,coord_defined,x,y,latitude,longitude,all_pce_active_bus,all_pde_active_bus,line_list,line_total_miles,load_list,section_id,total_customers
114,95217345002086b,"[1, 2, 3]",3,7.967434,3.917069,1,-49.318135,-16.602941,0.0,0.0,[Storage.battery_0],"[Line.cmt_5722432, Line.smt_55281851]","[LINE.cmt_5722432, LINE.smt_55281851]",0.0,[None],0,0
354,82033655002086,"[1, 2, 3]",3,7.967434,8.060878,1,-49.330809,-16.616380,0.0,0.0,[Storage.battery_1],[Line.smt_53072692],[LINE.smt_53072692],0.0,[None],0,0
1006,81489125002086,"[1, 2, 3]",3,7.967434,1.253358,1,-49.324749,-16.586983,0.0,0.0,[Storage.battery_2],"[Line.smt_53074008, Line.smt_53074009]","[LINE.smt_53074008, LINE.smt_53074009]",0.0,[None],0,0
1093,82021615286802,"[1, 2, 3, 4]",4,0.219393,4.759274,1,-49.311880,-16.604551,0.0,0.0,"[Load.m2_trf_5286802a, Load.m1_trf_5286802a, R...",[Transformer.trf_5286802a],[None],0.0,"[LOAD.m2_trf_5286802a, LOAD.m1_trf_5286802a]",0,0
1112,82025745286821,"[1, 2, 3, 4]",4,0.219393,8.165772,1,-49.330934,-16.628300,0.0,0.0,"[Load.m2_trf_5286821a, Load.m1_trf_5286821a, R...",[Transformer.trf_5286821a],[None],0.0,"[LOAD.m2_trf_5286821a, LOAD.m1_trf_5286821a]",0,0


Mapa com os barramentos

In [6]:
colors = ["red", "blue", "green", "purple", "yellow"]
bus_list = []
for i, bus in enumerate(solution_buses):
    bus_list.append(dss_tools.interactive_view.circuit_get_bus_marker(name=bus, color=colors[i], size=10)) 

#bus_list = [dss_tools.interactive_view.circuit_get_bus_marker(name=solution_buses[3], color=colors[3], size=10)]

print(bus_list)

dss_tools.interactive_view.circuit_geoplot(parameter="active power", title="Circuit Plot with Solution Buses", show=True, bus_markers=bus_list)

[CircuitBusMarker(name='95217345002086b', symbol='square', size=10, color='red', marker_name='95217345002086b'), CircuitBusMarker(name='82033655002086', symbol='square', size=10, color='blue', marker_name='82033655002086'), CircuitBusMarker(name='81489125002086', symbol='square', size=10, color='green', marker_name='81489125002086'), CircuitBusMarker(name='82021615286802', symbol='square', size=10, color='purple', marker_name='82021615286802'), CircuitBusMarker(name='82025745286821', symbol='square', size=10, color='yellow', marker_name='82025745286821')]


In [36]:
buses_df = dss_tools.model.buses_df
trafo_df = dss_tools.model.transformers_df
lines_df = dss_tools.model.lines_df

trafo_df["buses"] = trafo_df["buses"].apply(
    lambda x: [bus.strip() for bus in x.strip("[]").split(",") if bus.strip()]
)

bus_list = []
bus_in_loads = []
bus_graph = []

for bus in solution_buses:
    query_df = buses_df.query(f"name == '{bus}'")[["all_pce_active_bus", "all_pde_active_bus"]]
    pce_elements = query_df["all_pce_active_bus"].iloc[0]

    if "load" in pce_elements[0].lower():
        bus_in_loads.append(bus)
        pde_elements = query_df["all_pde_active_bus"].iloc[0]
        pde_element = pde_elements[0].split(".")[1]

        if "trf_" in pde_element.lower():
            buses = trafo_df.query(f"name == '{pde_element}'")["buses"].iloc[0]
            bus1 = buses[0]
            bus1 = bus1.split(".")[0]
            print(bus1)
            bus_graph.append(bus1)

        else:
            bus1 = lines_df.query(f"name == '{pde_element}'")["bus"].iloc[0]
            bus_graph.append(bus1)

for i, bus in enumerate(bus_graph):
    bus_list.append(dss_tools.interactive_view.circuit_get_bus_marker(name=bus, color=colors[i], size=10)) 

print(bus_list)

dss_tools.interactive_view.circuit_geoplot(parameter="active power", title="Circuit Plot with Solution Buses", show=True, bus_markers=bus_list)


82021615002086
82025745002086
[CircuitBusMarker(name='82021615002086', symbol='square', size=10, color='red', marker_name='82021615002086'), CircuitBusMarker(name='82025745002086', symbol='square', size=10, color='blue', marker_name='82025745002086')]


Barras ilhadas (só um elemento PDE conectado)

In [ ]:
print(type(buses_df["all_pce_active_bus"].iloc[1][0]))
print(type(buses_df["all_pde_active_bus"].iloc[1]))

isolated_buses = []

for _, row in buses_df.iterrows():
    if row["all_pce_active_bus"][0] == "None" and len(row["all_pde_active_bus"]) < 2:
        isolated_buses.append(row["name"])

print(isolated_buses)
print(len(isolated_buses))

#for bus in isolated_buses:
#    display(buses_df.query(f"name == '{bus}'"))


<class 'str'>
<class 'list'>
['i888888340825002086b', '82021905002086', '82031435002086', '82036215002086', '82032955002086', '82033655002086', '82032275002086', '82049135002086', '82050245002086', '82049695002086', '82040095002086', '82049745002086', '82049895002086', '82049925002086', '82040735002086', '82065615002086', '82039665002086', '82042055002086', '88685565002086', '88686185002086', '94074915002086b', '94074915002086', '94125525002086b', '94128575002086b', '94511165002086b', '94511165002086', '94522045002086', '94531885002086b', '94531885002086', '94324465002086b', '94324465002086', '94329955002086', '94329955002086b', '94335985002086', '94335985002086b', '94336115002086', '94336115002086b', '94660965002086', '94661005002086', '90349985002086', '95170945002086b', '95170975002086b', '95200805002086b', '95200805002086', '95217195002086', '95217195002086b', '95224855002086', '95225005002086', '94563035002086b', '94563035002086', '95242395002086', '95244945002086', '9477792500208

,name,distance,all_pce_active_bus,all_pde_active_bus
128,i888888340825002086b,0.0,[None],[Line.cmt_sa0002751]


,name,distance,all_pce_active_bus,all_pde_active_bus
134,82021905002086,3.916473,[None],[Line.smt_53072476]


,name,distance,all_pce_active_bus,all_pde_active_bus
291,82031435002086,7.005301,[None],[Line.smt_53072627]


,name,distance,all_pce_active_bus,all_pde_active_bus
302,82036215002086,6.082723,[None],[Line.smt_53072638]


...

,name,distance,all_pce_active_bus,all_pde_active_bus
1063,81480555002086b,8.819677,[None],[Line.smt_55501923]


,name,distance,all_pce_active_bus,all_pde_active_bus
1066,96957955002086b,8.832597,[None],[Line.smt_55501929]


,name,distance,all_pce_active_bus,all_pde_active_bus
1068,i888888516525002086g,0.0,[None],[Line.smt_88888869150]


,name,distance,all_pce_active_bus,all_pde_active_bus
1070,int157715002086,0.0,[None],[Line.smt_88888869156]


,name,distance,all_pce_active_bus,all_pde_active_bus
1071,i888888516585002086,0.0,[None],[Line.smt_88888869158]


In [21]:
bus_list = []
for i, bus in enumerate(isolated_buses):
    bus_list.append(dss_tools.interactive_view.circuit_get_bus_marker(name=bus, size=10)) 

print(bus_list)

dss_tools.interactive_view.circuit_geoplot(parameter="active power", title="Circuit Plot with Isolated Buses", show=True, bus_markers=bus_list)

[CircuitBusMarker(name='i888888340825002086b', symbol='square', size=10, color='black', marker_name='i888888340825002086b'), CircuitBusMarker(name='82021905002086', symbol='square', size=10, color='black', marker_name='82021905002086'), CircuitBusMarker(name='82031435002086', symbol='square', size=10, color='black', marker_name='82031435002086'), CircuitBusMarker(name='82036215002086', symbol='square', size=10, color='black', marker_name='82036215002086'), CircuitBusMarker(name='82032955002086', symbol='square', size=10, color='black', marker_name='82032955002086'), CircuitBusMarker(name='82033655002086', symbol='square', size=10, color='black', marker_name='82033655002086'), CircuitBusMarker(name='82032275002086', symbol='square', size=10, color='black', marker_name='82032275002086'), CircuitBusMarker(name='82049135002086', symbol='square', size=10, color='black', marker_name='82049135002086'), CircuitBusMarker(name='82050245002086', symbol='square', size=10, color='black', marker_nam